# การจำแนกรูปทรงใบหน้า (Face Shape Classification Model)
การพัฒนาแบบจำลองจำแนกประเภท และการแบ่งชุดข้อมูล (Data Splitting: Train 70% / Val 15% / Test 15%)

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [2]:
# 1. โหลดข้อมูลสัดส่วนจากภาพใบหน้าในโฟลเดอร์ faces/ (7 รูปทรงใบหน้า)
dataset_dir = Path('faces')
classes = sorted([d.name for d in dataset_dir.iterdir() if d.is_dir()])
print(f'Detected classes ({len(classes)}): {classes}')

# สกัด Features 4 ค่าหลักตามหลักกายวิภาค (โหนกแก้ม/กราม, อัตราส่วนความยาว, คาง, หน้าผาก)
print(f'Total samples: {len(X)}')


Detected classes (7): ['diamond', 'heart', 'oblong', 'oval', 'pear', 'round', 'square']
Total samples: 595


In [3]:
# 2. การแบ่งชุดข้อมูล (Data Splitting): Train 70% / Validation 15% / Test 15%
# ขั้นที่ 1: แยก Train 70% ออกมาก่อน (เหลือ 30% ไว้สำหรับ Val และ Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# ขั้นที่ 2: นำส่วน 30% มาแบ่งครึ่งเป็น Validation 15% และ Test 15%
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Training Set   (70%): {len(X_train)} samples')
print(f'Validation Set (15%): {len(X_val)} samples')
print(f'Test Set       (15%): {len(X_test)} samples')


Training Set   (70%): 416 samples
Validation Set (15%): 89 samples
Test Set       (15%): 90 samples


In [4]:
# 3. สร้างและฝึกฝนโมเดลด้วย Training Set พร้อมปรับแต่งค่าด้วย Validation Set
model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
model.fit(X_train, y_train)

# วัดผลเบื้องต้นบน Validation Set เพื่อคุม Overfitting
val_acc = accuracy_score(y_val, model.predict(X_val))
print(f'Validation Accuracy: {val_acc:.4f}')


Validation Accuracy: 0.8652


In [5]:
# 4. ทดสอบประสิทธิภาพขั้นสุดท้ายด้วย Test Set (ข้อมูลที่ไม่เคยผ่านการฝึกฝน)
answer = model.predict(X_test)

print(f'Score: {accuracy_score(y_test, answer):.4f}
')
print(classification_report(y_test, answer, target_names=classes))
print(confusion_matrix(y_test, answer))


Score: 0.8444

              precision    recall  f1-score   support

     diamond       0.86      1.00      0.92        12
       heart       0.91      0.77      0.83        13
      oblong       0.92      0.92      0.92        13
        oval       0.64      0.69      0.67        13
        pear       0.93      1.00      0.96        13
       round       0.79      0.85      0.81        13
      square       0.90      0.69      0.78        13

    accuracy                           0.84        90
   macro avg       0.85      0.85      0.84        90
weighted avg       0.85      0.84      0.84        90

[[12  0  0  0  0  0  0]
 [ 0 10  0  3  0  0  0]
 [ 0  0 12  1  0  0  0]
 [ 2  1  1  9  0  0  0]
 [ 0  0  0  0 13  0  0]
 [ 0  0  0  1  0 11  1]
 [ 0  0  0  0  1  3  9]]
